# 02 — Aedes Puerto Rico traps

Loads **field** matrices from `data/aedes/raw/` when present (`[EMPÍRICO]`).
Falls back to synthetic proxy under `data/aedes/proxy/` (`[OPERACIONAL]`).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'python').is_dir():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'python'))

from core import accumulate_time, compute_taus, load_aedes_sites


In [ ]:
result = load_aedes_sites(root=ROOT, prefer_raw=True)
print(result.label, 'source=', result.source)
if result.directory is not None:
    print('dir=', result.directory)

summaries = []
for name, X in result.sites.items():
    X = np.asarray(X, dtype=float)
    tg, _ = compute_taus(X, window_size=13)
    T, _, g, depth = accumulate_time(tg, window_size=13)
    valid = tg[~np.isnan(tg)]
    summaries.append((name, X, tg, T, g, depth, valid))
    print(f'{name}: shape={X.shape} mean_tau={valid.mean():.3f} T_RECD={T[-1]:.5f} max_k={int(depth.max())}')


In [ ]:
n = len(summaries)
fig, axes = plt.subplots(n, 2, figsize=(10, 3.2 * n), squeeze=False)
for i, (name, X, tg, T, g, depth, valid) in enumerate(summaries):
    axes[i, 0].plot(tg, label='τₛ')
    axes[i, 0].axhline(0.41, ls='--', c='C1', alpha=0.7)
    axes[i, 0].axhline(-0.41, ls='--', c='C1', alpha=0.7)
    axes[i, 0].set_title(f'{name} · τₛ')
    axes[i, 0].legend(loc='upper right')
    axes[i, 1].plot(T, label='T_RECD')
    axes[i, 1].set_title(f'{name} · RECD clock')
    axes[i, 1].legend(loc='upper left')
title = f'{result.label} · source={result.source}'
fig.suptitle(title, y=1.01)
fig.tight_layout()
plt.show()
